# UsedCarAI — 04 Hybrid Appraisal




In [ ]:
from pathlib import Path
import os
import sys

# Discover CPE-306 when opened from the project or UsedcarAi directory.
# Set USEDCAR_PROJECT_DIR to use a different checkout location.
_candidates = (Path.cwd(), *Path.cwd().parents)
_default_project = next(
    (p for p in _candidates if (p / "dataset").is_dir() and (p / "app").is_dir()),
    Path("/home/regen/Works/CPE-306"),
)
PROJECT_DIR = Path(os.environ.get("USEDCAR_PROJECT_DIR", _default_project)).expanduser().resolve()
DATASET_DIR = Path(os.environ.get("DATASET_DIR", PROJECT_DIR / "dataset")).expanduser().resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Use the Python kernel from the project's ML environment.")


In [ ]:
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = DATASET_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RF_MODEL_DIR = PROJECT_ROOT / "models" / "random_forest"
LR_MODEL_DIR = PROJECT_ROOT / "models" / "linear_regression"
MODEL_DIR = PROJECT_ROOT / "models" / "hybrid"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
PRED_DIR = PROJECT_ROOT / "results" / "predictions"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

for d in [MODEL_DIR, METRICS_DIR, PRED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for p in [
    PROCESSED_DIR / "train.csv",
    PROCESSED_DIR / "validation.csv",
    PROCESSED_DIR / "test.csv",
    RF_MODEL_DIR / "random_forest_pipeline.joblib",
    LR_MODEL_DIR / "linear_regression_pipeline.joblib",
]:
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์ที่จำเป็น: {p}")

print("Project:", PROJECT_ROOT)

## Step 1 — โหลด Dataset และ Models

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

rf_model = joblib.load(RF_MODEL_DIR / "random_forest_pipeline.joblib")
lr_model = joblib.load(LR_MODEL_DIR / "linear_regression_pipeline.joblib")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

## Step 2 — Configuration

Hybrid ใช้:
- ML prediction = Random Forest
- Comparable price = median/weighted median ของรถใน Train ที่ใกล้เคียง
- `ML_WEIGHT = 0.50`
- `COMP_WEIGHT = 0.50`
- ถ้า Comparable น้อยกว่า `MIN_COMPS` จะ fallback ไปใช้ ML 100%

Comparable จะพยายาม match ตามลำดับ:
1. brand + model + sub_model
2. แล้วกรอง/ให้น้ำหนักปี, mileage, engine_size, fuel_type, transmission, body_type
3. ถ้ากลุ่มรุ่นย่อยไม่มีข้อมูลเพียงพอ จะ fallback ไป brand + model


In [ ]:
ML_WEIGHT = 0.50
COMP_WEIGHT = 0.50

MIN_COMPS = 3
MAX_COMPS = 20

YEAR_WINDOW = 5
MILEAGE_WINDOW = 100_000

# น้ำหนักความคล้ายสำหรับ Comparable
W_YEAR = 0.30
W_MILEAGE = 0.30
W_ENGINE = 0.15
W_FUEL = 0.10
W_TRANS = 0.10
W_BODY = 0.05

assert abs(ML_WEIGHT + COMP_WEIGHT - 1.0) < 1e-12

## Step 3 — Random Forest / Linear Regression Predictions

In [ ]:
TEST_FEATURES = [
    "brand", "model", "sub_model",
    "fuel_type", "transmission", "body_type",
    "color", "province", "location", "seller_type",
    "model_year", "mileage", "engine_size", "number_of_seats",
    "vehicle_age", "vehicle_age_sq", "mileage_log1p", "mileage_per_year"
]

X_test = test_df[TEST_FEATURES]

rf_pred = rf_model.predict(X_test)
lr_pred = lr_model.predict(X_test)

print("RF predictions:", len(rf_pred))
print("LR predictions:", len(lr_pred))

## Step 4 — Comparable Distance Functions

In [ ]:
def numeric_similarity(diff, scale):
    if pd.isna(diff) or scale <= 0:
        return 0.0
    return max(0.0, 1.0 - float(diff) / float(scale))

def comparable_score(query, candidates):
    q_year = query["model_year"]
    q_mileage = query["mileage"]
    q_engine = query["engine_size"]

    year_sim = np.maximum(
        0.0,
        1.0 - np.abs(candidates["model_year"].to_numpy(dtype=float) - q_year) / YEAR_WINDOW
    )

    mileage_sim = np.maximum(
        0.0,
        1.0 - np.abs(candidates["mileage"].to_numpy(dtype=float) - q_mileage) / MILEAGE_WINDOW
    )

    if pd.isna(q_engine):
        engine_sim = np.full(len(candidates), 0.5)
    else:
        c_engine = candidates["engine_size"].to_numpy(dtype=float)
        engine_sim = np.where(
            np.isnan(c_engine),
            0.5,
            np.maximum(0.0, 1.0 - np.abs(c_engine - q_engine) / max(abs(float(q_engine)), 1.0))
        )

    fuel_sim = (
        candidates["fuel_type"].fillna("").astype(str).to_numpy()
        == str(query["fuel_type"]) if pd.notna(query["fuel_type"])
        else np.full(len(candidates), 0.5)
    ).astype(float)

    trans_sim = (
        candidates["transmission"].fillna("").astype(str).to_numpy()
        == str(query["transmission"]) if pd.notna(query["transmission"])
        else np.full(len(candidates), 0.5)
    ).astype(float)

    body_sim = (
        candidates["body_type"].fillna("").astype(str).to_numpy()
        == str(query["body_type"]) if pd.notna(query["body_type"])
        else np.full(len(candidates), 0.5)
    ).astype(float)

    score = (
        W_YEAR * year_sim +
        W_MILEAGE * mileage_sim +
        W_ENGINE * engine_sim +
        W_FUEL * fuel_sim +
        W_TRANS * trans_sim +
        W_BODY * body_sim
    )

    return score

## Step 5 — Build Comparable Reference จาก Train เท่านั้น

In [ ]:
# Group indexes เพื่อค้นหา candidate เร็วขึ้น
group_exact = {}
group_model = {}

for idx, row in train_df.reset_index(drop=True).iterrows():
    brand = str(row["brand"])
    model = str(row["model"])
    sub = "" if pd.isna(row["sub_model"]) else str(row["sub_model"])

    key_exact = (brand, model, sub)
    key_model = (brand, model)

    group_exact.setdefault(key_exact, []).append(idx)
    group_model.setdefault(key_model, []).append(idx)

train_lookup = train_df.reset_index(drop=True)

print("Exact groups:", len(group_exact))
print("Brand+Model groups:", len(group_model))

### Comparable logic

สำหรับรถแต่ละคันใน Test:
1. หารถใน Train ที่ `brand + model + sub_model` ตรงกัน
2. ถ้าไม่พอ ใช้ `brand + model`
3. จำกัดปีและเลขไมล์ที่ใกล้เคียงเมื่อหาได้
4. คำนวณ similarity score
5. เลือก Comparable สูงสุดไม่เกิน `MAX_COMPS`
6. ใช้ weighted median/average ของราคา Comparable
7. ถ้าไม่มี Comparable เพียงพอ → fallback เป็น RF prediction


In [ ]:
def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    cumulative = np.cumsum(weights)
    cutoff = weights.sum() / 2.0
    return float(values[np.searchsorted(cumulative, cutoff)])

def find_comparable(query):
    brand = str(query["brand"])
    model = str(query["model"])
    sub = "" if pd.isna(query["sub_model"]) else str(query["sub_model"])

    exact_key = (brand, model, sub)
    model_key = (brand, model)

    if exact_key in group_exact:
        idxs = group_exact[exact_key]
    else:
        idxs = group_model.get(model_key, [])

    if not idxs:
        return np.nan, 0, 0.0

    candidates = train_lookup.iloc[idxs].copy()

    # Hard proximity filters when enough candidates exist
    if pd.notna(query["model_year"]):
        filtered = candidates[
            candidates["model_year"].between(
                float(query["model_year"]) - YEAR_WINDOW,
                float(query["model_year"]) + YEAR_WINDOW
            )
        ]
        if len(filtered) >= MIN_COMPS:
            candidates = filtered

    if pd.notna(query["mileage"]):
        filtered = candidates[
            candidates["mileage"].between(
                max(0, float(query["mileage"]) - MILEAGE_WINDOW),
                float(query["mileage"]) + MILEAGE_WINDOW
            )
        ]
        if len(filtered) >= MIN_COMPS:
            candidates = filtered

    if len(candidates) == 0:
        return np.nan, 0, 0.0

    scores = comparable_score(query, candidates)
    candidates = candidates.copy()
    candidates["similarity"] = scores

    candidates = candidates.sort_values("similarity", ascending=False).head(MAX_COMPS)

    if len(candidates) < MIN_COMPS:
        return np.nan, len(candidates), float(candidates["similarity"].mean()) if len(candidates) else 0.0

    prices = candidates["price"].to_numpy(dtype=float)
    weights = np.clip(candidates["similarity"].to_numpy(dtype=float), 0.01, None)

    comp_price = weighted_median(prices, weights)
    avg_similarity = float(weights.mean())

    return comp_price, len(candidates), avg_similarity

## Step 6 — Generate Hybrid Predictions

In [ ]:
records = []

t0 = time.time()

for i, (_, row) in enumerate(test_df.iterrows()):
    comp_price, n_comp, comp_similarity = find_comparable(row)

    if pd.notna(comp_price) and n_comp >= MIN_COMPS:
        hybrid_price = (
            ML_WEIGHT * rf_pred[i] +
            COMP_WEIGHT * comp_price
        )
        used_hybrid = True
    else:
        hybrid_price = rf_pred[i]
        used_hybrid = False

    records.append({
        "listing_id": row["listing_id"],
        "brand": row["brand"],
        "model": row["model"],
        "sub_model": row["sub_model"],
        "model_year": row["model_year"],
        "mileage": row["mileage"],
        "actual_price": row["price"],
        "rf_predicted_price": rf_pred[i],
        "linear_regression_predicted_price": lr_pred[i],
        "comparable_price": comp_price,
        "comparable_count": n_comp,
        "comparable_similarity": comp_similarity,
        "hybrid_predicted_price": hybrid_price,
        "hybrid_used": used_hybrid,
    })

hybrid_df = pd.DataFrame(records)

print(f"Comparable calculation time: {(time.time() - t0)/60:.2f} min")
print("Hybrid used:", int(hybrid_df["hybrid_used"].sum()))
print("RF fallback:", int((~hybrid_df["hybrid_used"]).sum()))

display(hybrid_df.head())

## Step 7 — Evaluate RF vs Linear Regression vs Hybrid

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return float(
        100 * np.mean(
            2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]
        )
    )

def metric_dict(y_true, y_pred):
    return {
        "MAE_THB": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_THB": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "sMAPE_percent": smape(y_true, y_pred),
        "MedianAE_THB": float(median_absolute_error(y_true, y_pred)),
    }

y_test = test_df["price"]

rf_metrics = metric_dict(y_test, hybrid_df["rf_predicted_price"])
lr_metrics = metric_dict(y_test, hybrid_df["linear_regression_predicted_price"])
hy_metrics = metric_dict(y_test, hybrid_df["hybrid_predicted_price"])

comparison = pd.DataFrame([
    {"model": "Random Forest", **rf_metrics},
    {"model": "Linear Regression", **lr_metrics},
    {"model": "Hybrid 50/50", **hy_metrics},
])

display(comparison)

comparison.to_csv(
    METRICS_DIR / "model_comparison_test.csv",
    index=False,
    encoding="utf-8-sig"
)

## Step 8 — Error by Price Band

In [ ]:
def price_band_metrics(pred_column):
    tmp = hybrid_df[["actual_price", pred_column]].copy()
    bins = [-np.inf, 300_000, 600_000, 1_000_000, np.inf]
    labels = ["<300k", "300k-600k", "600k-1m", ">1m"]
    tmp["price_band"] = pd.cut(tmp["actual_price"], bins=bins, labels=labels)

    rows = []
    for band, g in tmp.groupby("price_band", observed=True):
        m = metric_dict(g["actual_price"], g[pred_column])
        rows.append({"price_band": str(band), "n": len(g), **m})
    return pd.DataFrame(rows)

hybrid_band = price_band_metrics("hybrid_predicted_price")
display(hybrid_band)

hybrid_band.to_csv(
    METRICS_DIR / "hybrid_test_by_price_band.csv",
    index=False,
    encoding="utf-8-sig"
)

## Step 9 — Comparable Coverage

In [ ]:
coverage = pd.DataFrame([{
    "test_rows": len(hybrid_df),
    "hybrid_used_rows": int(hybrid_df["hybrid_used"].sum()),
    "fallback_to_rf_rows": int((~hybrid_df["hybrid_used"]).sum()),
    "hybrid_coverage_percent": float(hybrid_df["hybrid_used"].mean() * 100),
    "mean_comparable_count": float(hybrid_df["comparable_count"].mean()),
    "median_comparable_count": float(hybrid_df["comparable_count"].median()),
    "mean_similarity": float(hybrid_df["comparable_similarity"].replace(0, np.nan).mean())
}])

display(coverage)

coverage.to_csv(
    METRICS_DIR / "hybrid_comparable_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

## Step 10 — Save Predictions and Model Configuration

In [ ]:
hybrid_output = PRED_DIR / "hybrid_test_predictions.csv"
hybrid_df.to_csv(hybrid_output, index=False, encoding="utf-8-sig")

metadata = {
    "model": "Hybrid",
    "ml_component": "Random Forest",
    "comparable_component": "Train-only Comparable Market Reference",
    "ml_weight": ML_WEIGHT,
    "comparable_weight": COMP_WEIGHT,
    "min_comparables": MIN_COMPS,
    "max_comparables": MAX_COMPS,
    "year_window": YEAR_WINDOW,
    "mileage_window": MILEAGE_WINDOW,
    "test_metrics": hy_metrics,
    "comparable_coverage_percent": float(hybrid_df["hybrid_used"].mean() * 100),
    "note": "Comparable reference is built from training data only to avoid target leakage."
}

# Hybrid does not require a second trained estimator. Save configuration and reusable references.
# Save a compact market reference containing the training rows used by the comparable engine.
market_reference = train_df[[
    "brand", "model", "sub_model", "model_year", "mileage",
    "fuel_type", "transmission", "body_type", "engine_size", "price"
]].copy()

market_reference.to_csv(
    MODEL_DIR / "market_reference_train_only.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(MODEL_DIR / "hybrid_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved predictions:", hybrid_output)
print("Saved market reference:", MODEL_DIR / "market_reference_train_only.csv")
print("Saved metadata:", MODEL_DIR / "hybrid_metadata.json")

## Step 11 — Hybrid Visualization

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(
    hybrid_df["actual_price"],
    hybrid_df["hybrid_predicted_price"],
    s=8,
    alpha=0.35
)

lo = float(min(
    hybrid_df["actual_price"].min(),
    hybrid_df["hybrid_predicted_price"].min()
))
hi = float(max(
    hybrid_df["actual_price"].max(),
    hybrid_df["hybrid_predicted_price"].max()
))

plt.plot([lo, hi], [lo, hi])
plt.xlabel("Actual price (THB)")
plt.ylabel("Hybrid predicted price (THB)")
plt.title("Hybrid — Actual vs Predicted")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "hybrid_actual_vs_predicted.png",
    dpi=180
)
plt.show()

# Step 4 Complete

ควรมี:

```text
models/hybrid/
├── hybrid_metadata.json
└── market_reference_train_only.csv

results/metrics/
├── model_comparison_test.csv
├── hybrid_test_by_price_band.csv
└── hybrid_comparable_coverage.csv

results/predictions/
└── hybrid_test_predictions.csv

results/figures/
└── hybrid_actual_vs_predicted.png
```

### สำคัญ
ค่าผลลัพธ์ของ Hybrid ยัง **ไม่ควรถูกเรียกว่าเป็นโมเดลที่ดีที่สุด** จาก Notebook นี้อย่างเดียว

ต้องไป Step 5 เพื่อ:
- รวม metrics ของทั้ง 3 วิธี
- ตรวจว่าทั้ง 3 ใช้ Test ชุดเดียวกัน
- เปรียบเทียบ error
- วิเคราะห์ตามช่วงราคา
- จัดทำกราฟ/ตารางสำหรับรายงาน

ขั้นต่อไปคือ `05_model_comparison.ipynb`
